<a href="https://colab.research.google.com/github/Abdulwaliy/Flyrank-ML-Internship/blob/main/Abdulwaliy/Flyrank-ML-Internship/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulwaliy/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This lane is fundamentally a scoring and ranking task, not classification. The output is a continuous opportunity score per page — how far below its position tier's expected CTR it sits, weighted by traffic volume — which is then sorted into a ranked review list. Classification is a plausible future extension (e.g. predicting whether a flagged page's CTR improves after review), but it would require a leakage-safe label built from a future time window, which the current data doesn't yet support. Clustering was considered but rejected: it answers "what kinds of pages are there" rather than "which pages deserve attention now," which doesn't serve the actual decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

This lane does not predict a learned target — it produces a defined rule/proxy score, not an observed-outcome label. ctr_gap_pct and opportunity_score are formulas built from data already observed in the current window; neither looks forward in time. This is a deliberate scope choice, not an oversight: I'm aware the repo's pipeline defines its own label, is_declining_label (1 when trend_direction == "down"), but per the data dictionary this label is itself a defined rule applied to trend_pct — not an independently observed outcome — so it isn't something I'd treat as ground truth either. If I later wanted a true observed-outcome target for this lane, it would need to be something like "did CTR improve in the 30 days after a page was flagged and edited" — an outcome I don't currently have, since it requires a before/after intervention record the starter dataset doesn't contain.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*


Because I don't yet have an observed-outcome label (see previous section), I can't defend a metric like precision or recall against ground truth. Instead, my week-1 success metric is opportunity concentration: the share of total opportunity score (gap × volume) captured by the top 5% of ranked pages. If concentration is well above 5% (an even, non-informative spread), that's evidence the ranking meaningfully prioritizes reviewer attention rather than just reordering noise. This is a defensible, computable-today metric — not a substitute for outcome-based validation, which I'll pursue once I can define a leakage-safe label (e.g. did CTR improve after a page was reviewed) in a later week.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one page (content_id) — not a client, not a day, not a query. Each row in the starter dataset is a single content item aggregated over a trailing 90-day window, and my derived columns (ctr_gap_pct, opportunity_score) preserve that grain rather than rolling up to client or down to daily. This matters because the lane's output — "review this page" — only makes sense as an action if each row maps to exactly one reviewable thing. client_id appears only for grouping (e.g. client-holdout splits later), never as the unit being scored.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
#df.head()

print(f"Total rows: {len(df)}")
print(f"Unique content_id: {df['content_id'].nunique()}")
print(f"Rows == unique content_id? {len(df) == df['content_id'].nunique()}")

# Show what one row actually looks like
tier_expected = df.groupby('position_tier')['ctr'].transform('mean')
df['ctr_gap_pct'] = (df['ctr'] - tier_expected) / tier_expected
df['opportunity_score'] = (-df['ctr_gap_pct']).clip(lower=0) * df['impressions_last_30d']

print(df.columns.tolist())


print(f"Total rows: {len(df)}")
print(f"Unique content_id: {df['content_id'].nunique()}")
print(f"Rows == unique content_id? {len(df) == df['content_id'].nunique()}")

df[['content_id', 'client_id', 'position_tier', 'ctr', 'impressions_last_30d',
    'opportunity_score']].head(3)


Total rows: 30000
Unique content_id: 30000
Rows == unique content_id? True
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'ctr_gap_pct', 'opportunity_score']
Total rows: 30000
Unique content_id: 30000
Rows == unique content_id? True


,content_id,client_id,position_tier,ctr,impressions_last_30d,opportunity_score
0,content_304f48230142,client_f369cb89fc,striking,0.76,578,0.000000
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,0.05,2501,1938.937414
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,0.09,2382,1418.425613


In [6]:
len(df) == df['content_id'].nunique()

True

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule struggles here for three concrete reasons, not hypothetical ones. First, a single CTR-gap threshold cannot separate a real problem from a page that simply lacks enough data — pages in the top_3 tier have a median of just 53 impressions in 90 days, where the data dictionary notes a single click can swing CTR by ~2 percentage points, so a 0.00% CTR there is often noise, not failure, and a fixed rule treats both identically. Second, I measured this noise directly: low-volume pages (under 500 impressions in the last 30 days) show a CTR-gap standard deviation nearly 10x larger than high-volume pages (7.89 vs 0.81), and 86% of low-volume pages show an extreme gap versus 65% of high-volume pages — meaning any single fixed threshold will either over-flag noisy low-volume pages or under-flag real high-volume problems, with no one cutoff working for both. Third, my reason codes (low CTR for position, strong position, weak engagement, stale content) interact rather than add independently, and a hand-written if/elif chain forces an arbitrary, unvalidated choice about which combination matters most when several are true at once.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.